# Milestone 2: Data Quality, Cleaning, dan Validasi

Lanjutan dari `01_data_profiling.ipynb`. Rencana penanganan berdasarkan temuan Milestone 1:

| Dimensi kualitas | Masalah | Tindakan |
|---|---|---|
| Completeness | Tidak ada null eksplisit, tapi ada 189 `frp = 0` dan 3 `bright_ti5 ≤ 150 K` | Ubah jadi null. FRP diimputasi median grup, `bright_ti5` dibiarkan null |
| Completeness (temporal) | 55 hari tanpa deteksi Suomi NPP, 7 hari NOAA-20 | Identifikasi rentangnya, simpan sebagai tabel hari-tanpa-data untuk Milestone 3 |
| Uniqueness | Potensi deteksi ganda dan NRT yang sudah digantikan SP | Deduplikasi pada kunci deteksi, SP diprioritaskan (DuckDB) |
| Validity / Accuracy | FRP sangat menceng, saturasi I4, `ti4 < ti5`, confidence `low` | Outlier dideteksi dengan IQR skala log. Ditandai (flag) dan dibuat kolom winsorized, **nilai asli dipertahankan** |
| Consistency | Nama provinsi bahasa Inggris, waktu masih UTC, tipe kategorik masih string | Pemetaan ke nama provinsi Indonesia + pulau + zona waktu, waktu lokal, tipe `Date`/`Enum` |

Pembagian kerja tools mengikuti materi P03–P04: **DuckDB** untuk profiling, operasi berat satu pass penuh (deduplikasi), dan validasi SQL. **Polars LazyFrame** untuk transformasi kolom yang dieksekusi streaming (`sink_parquet`). Kode pipeline ada di `src/cleaning.py`, aturan validasi di `src/validation.py`.

In [1]:
import sys
import tempfile
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import duckdb
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl

from src import cleaning, validation
from src.config import CLEAN_PATH, DAILY_REF_PATH, FIGURES_DIR, GAP_PATH, RAW_PATH

pio.renderers.default = "notebook_connected"
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(60)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {RAW_PATH}. Lihat data/README.md.")

# Batasi memori DuckDB agar aman di laptop RAM 4-8 GB. Kelebihan data di-spill ke folder temp.
connection = duckdb.connect()
connection.execute(f"""
    SET memory_limit = '1GB';
    SET preserve_insertion_order = false;
    SET temp_directory = '{Path(tempfile.gettempdir(), "duckdb_tmp").as_posix()}';
""")
connection.execute(f"CREATE OR REPLACE VIEW hotspot_raw AS SELECT * FROM read_parquet('{RAW_PATH.as_posix()}')")

jumlah_awal = connection.execute("SELECT COUNT(*) FROM hotspot_raw").fetchone()[0]
print(f"Data mentah: {jumlah_awal:,} baris")

Data mentah: 6,087,130 baris


## 1. Profiling dengan DuckDB `SUMMARIZE`

In [2]:
connection.execute("SUMMARIZE hotspot_raw").pl().select(
    "column_name", "column_type", "min", "max", "approx_unique", "null_percentage"
)

column_name,column_type,min,max,approx_unique,null_percentage
str,str,str,str,i64,"decimal[9,2]"
"""acq_date""","""TIMESTAMP_NS""","""2012-01-20 00:00:00""","""2026-10-27 00:00:00""",5255,0.00
"""acq_datetime""","""TIMESTAMP_NS""","""2012-01-20 03:57:00""","""2026-10-27 17:58:00""",147125,0.00
"""satellite""","""VARCHAR""","""NOAA-20""","""Suomi NPP""",2,0.00
"""latitude""","""DOUBLE""","""-10.97676""","""5.8782""",1269739,0.00
"""longitude""","""DOUBLE""","""95.01992""","""141.03944""",2579117,0.00
"""province""","""VARCHAR""","""Aceh""","""West Sumatra""",30,0.00
"""confidence""","""VARCHAR""","""high""","""nominal""",3,0.00
"""frp""","""DOUBLE""","""0.0""","""1951.3""",16788,0.00
"""bright_ti4""","""DOUBLE""","""207.93""","""367.0""",6877,0.00


### Laporan kualitas sebelum cleaning

Aturan validasi yang sama nanti dijalankan ulang pada data bersih. Di sini aturan dijalankan pada data mentah untuk melihat kondisi awal.

In [3]:
laporan_sebelum = validation.run_rules(
    connection, "hotspot_raw", validation.RAW_RULES, unique_key=cleaning.KEY_COLUMNS
)
laporan_sebelum

aturan,dimensi,deskripsi,pelanggaran,persen,lulus
str,str,str,i64,f64,bool
"""kolom_kunci_lengkap""","""completeness""","""Satelit, waktu, dan koordinat wajib terisi""",0,0.0,true
"""frp_terisi""","""completeness""","""FRP wajib terisi""",0,0.0,true
"""frp_positif""","""validity""","""FRP deteksi api harus > 0 MW""",189,0.0031,false
"""bright_ti5_wajar""","""validity""","""Suhu kecerahan I5 <= 150 K tidak realistis""",3,0.0,false
"""koordinat_di_indonesia""","""validity""","""Koordinat berada di kotak wilayah Indonesia""",0,0.0,true
"""satelit_valid""","""validity""","""Hanya Suomi NPP / NOAA-20""",0,0.0,true
"""confidence_valid""","""validity""","""Confidence hanya low/nominal/high""",0,0.0,true
"""daynight_valid""","""validity""","""Siang/malam hanya D/N""",0,0.0,true
"""processing_valid""","""validity""","""Status pemrosesan hanya SP/NRT""",0,0.0,true


## 2. Missing Values

### 2a. Null eksplisit dan nilai yang secara makna "hilang"

Tidak ada null eksplisit. Tetapi `frp = 0` tidak bermakna untuk sebuah deteksi api (sensor tetap mendeteksi piksel panas, jadi pengukuran intensitasnya yang gagal), dan `bright_ti5 ≤ 150 K` jauh di bawah suhu permukaan bumi. Keduanya diperlakukan sebagai **missing**.

In [4]:
connection.execute("""
    SELECT satellite, daynight, confidence,
           COUNT(*) FILTER (WHERE frp <= 0)          AS frp_nol,
           COUNT(*) FILTER (WHERE bright_ti5 <= 150) AS ti5_tidak_wajar
    FROM hotspot_raw
    WHERE frp <= 0 OR bright_ti5 <= 150
    GROUP BY ALL
    ORDER BY frp_nol DESC
""").pl()

satellite,daynight,confidence,frp_nol,ti5_tidak_wajar
str,str,str,i64,i64
"""Suomi NPP""","""D""","""low""",73,0
"""NOAA-20""","""D""","""low""",33,0
"""Suomi NPP""","""D""","""high""",26,0
"""NOAA-20""","""D""","""high""",17,0
"""Suomi NPP""","""D""","""nominal""",13,0
"""NOAA-20""","""N""","""nominal""",8,0
"""Suomi NPP""","""N""","""nominal""",6,3
"""NOAA-20""","""N""","""high""",5,0
"""Suomi NPP""","""N""","""high""",5,0


**Strategi:**
- **FRP** dipakai di analisis intensitas, jadi diimputasi dengan **median FRP per (satelit, siang/malam, provinsi)**. Median dipilih karena distribusi FRP sangat menceng, dan kelompok dipakai karena FRP berbeda antar satelit, siang/malam, dan wilayah (pola yang sama dengan `fill_null(mean().over(...))` di P04). Baris yang diimputasi ditandai `frp_imputasi = True`. Jumlahnya hanya 189 baris (0,003%), jadi dampaknya ke statistik sangat kecil.
- **`bright_ti5`** tidak dipakai untuk analisis utama dan hanya 3 baris, jadi cukup dijadikan null tanpa imputasi. Barisnya tetap dipertahankan karena deteksi apinya sendiri valid.

Median per kelompok dihitung sekali sebagai tabel kecil, lalu digabung ke pipeline lewat join:

In [5]:
lazy_raw = pl.scan_parquet(RAW_PATH)
frp_medians = cleaning.frp_group_medians(lazy_raw)
print(f"{frp_medians.height} kelompok (satelit × siang/malam × provinsi)")
frp_medians.sort("frp_median_grup").head(5)

136 kelompok (satelit × siang/malam × provinsi)


satellite,daynight,province,frp_median_grup
str,str,str,f64
"""NOAA-20""","""N""","""Bengkulu""",0.79
"""Suomi NPP""","""N""","""Bengkulu""",0.88
"""Suomi NPP""","""N""","""West Java""",1.02
"""NOAA-20""","""N""","""West Java""",1.07
"""NOAA-20""","""N""","""Jakarta Special Capital Region""",1.07


### 2b. Missing temporal: hari tanpa deteksi

Untuk data deret waktu, "hari tanpa baris" bisa berarti dua hal: memang tidak ada api, atau satelit tidak beroperasi (data hilang). Rentang hari kosong yang berurutan dicari dengan teknik *gaps and islands* (selisih tanggal dengan `ROW_NUMBER()`):

In [6]:
hari_kosong = connection.execute("""
    WITH hari AS (
        SELECT satellite, acq_date::DATE AS tanggal FROM hotspot_raw GROUP BY ALL
    ),
    kalender AS (
        SELECT satellite,
               UNNEST(generate_series(MIN(tanggal), MAX(tanggal), INTERVAL 1 DAY))::DATE AS tanggal
        FROM hari GROUP BY satellite
    ),
    kosong AS (
        SELECT k.satellite, k.tanggal FROM kalender k ANTI JOIN hari h USING (satellite, tanggal)
    )
    SELECT *,
           tanggal - (ROW_NUMBER() OVER (PARTITION BY satellite ORDER BY tanggal))::INT AS grup
    FROM kosong
""").pl()

rentang_kosong = (
    hari_kosong.group_by("satellite", "grup")
    .agg(
        pl.col("tanggal").min().alias("mulai"),
        pl.col("tanggal").max().alias("selesai"),
        pl.len().alias("jumlah_hari"),
    )
    .drop("grup")
    .with_columns(pl.col("mulai").dt.month().is_between(6, 10).alias("di_musim_kemarau"))
    .sort("jumlah_hari", "mulai", descending=[True, False])
)
print(hari_kosong.group_by("satellite").len())
rentang_kosong.head(10)

shape: (2, 2)
┌───────────┬─────┐
│ satellite ┆ len │
│ ---       ┆ --- │
│ str       ┆ u32 │
╞═══════════╪═════╡
│ NOAA-20   ┆ 7   │
│ Suomi NPP ┆ 55  │
└───────────┴─────┘


satellite,mulai,selesai,jumlah_hari,di_musim_kemarau
str,date,date,u32,bool
"""Suomi NPP""",2022-07-27,2022-08-10,15,true
"""Suomi NPP""",2024-07-10,2024-07-15,6,true
"""Suomi NPP""",2026-07-11,2026-07-16,6,true
"""Suomi NPP""",2024-05-30,2024-06-03,5,false
"""Suomi NPP""",2024-07-24,2024-07-28,5,true
"""NOAA-20""",2023-07-19,2023-07-22,4,true
"""Suomi NPP""",2024-11-03,2024-11-05,3,false
"""Suomi NPP""",2026-03-10,2026-03-12,3,false
"""Suomi NPP""",2026-04-28,2026-04-30,3,false


Rentang terpanjang adalah **Suomi NPP 27 Juli – 10 Agustus 2022 (15 hari)**, pas di tengah musim kemarau. Tanggal ini sama persis dengan gangguan satelit yang diumumkan NASA: S-NPP mengalami anomali sejak 26 Juli 2022 dan produk VIIRS baru normal kembali 11 Agustus 2022 ([PO.DAAC, 19 Agu 2022](https://podaac.jpl.nasa.gov/announcements/2022-08-19-NOAA-and-NAVO-S-NPP-VIIRS-products-returned-to-normal-on-August-11-2022)). Rentang 10–15 Juli 2024 juga bertepatan dengan anomali produk VIIRS Suomi NPP pada Juli 2024 ([NASA Earthdata](https://www.earthdata.nasa.gov/data/alerts-outages/suomi-npp-viirs-data-product-anomaly-july-2024)). Hari-hari ini adalah **data hilang, bukan nol api**. Rentang kosong lain yang jatuh di musim kemarau juga hampir pasti gangguan data, karena mustahil seluruh Indonesia bebas api selama beberapa hari di bulan Juni–Oktober.

**Strategi:** hari kosong tidak diisi nol. Daftarnya disimpan ke `data/processed/hari_tanpa_deteksi.parquet`, supaya pada agregasi harian di Milestone 3 hari-hari ini bisa dibuat `null` (bukan 0) atau diisi dari satelit NOAA-20 yang tetap beroperasi.

In [7]:
GAP_PATH.parent.mkdir(parents=True, exist_ok=True)
hari_kosong.drop("grup").write_parquet(GAP_PATH)
print(f"Tersimpan: {GAP_PATH.relative_to(ROOT)} ({hari_kosong.height} baris)")

Tersimpan: data/processed/hari_tanpa_deteksi.parquet (62 baris)


## 3. Duplikat dan Tumpang Tindih NRT/SP (Tahap 1 — DuckDB)

Kunci satu deteksi: `(satellite, acq_datetime, latitude, longitude)`. Data NRT (sementara) seharusnya digantikan arsip final SP. Jika keduanya ada untuk tanggal yang sama, SP dipertahankan.

Deduplikasi butuh tabel hash untuk seluruh 6 juta baris. Percobaan dengan `LazyFrame.unique()` di mesin RAM 4 GB berakhir *out-of-memory*, jadi tahap ini dikerjakan DuckDB, yang bisa dibatasi memorinya dan melakukan spill ke disk. Hasilnya ditulis ke Parquet sementara.

In [8]:
STAGE_PATH = Path(tempfile.gettempdir(), "hotspots_stage_dedup.parquet")

mulai = time.perf_counter()
jumlah_stage = cleaning.dedup_with_duckdb(connection, RAW_PATH, STAGE_PATH)
durasi_stage = time.perf_counter() - mulai

print(f"Sebelum : {jumlah_awal:,} baris")
print(f"Sesudah : {jumlah_stage:,} baris")
print(f"Dibuang : {jumlah_awal - jumlah_stage:,} baris (duplikat + NRT tergantikan)")
print(f"Durasi  : {durasi_stage:.1f} detik")

Sebelum : 6,087,130 baris
Sesudah : 6,087,130 baris
Dibuang : 0 baris (duplikat + NRT tergantikan)
Durasi  : 9.2 detik


Tidak ada baris yang terbuang pada versi data ini, karena pembuat dataset sudah mengganti NRT dengan SP. Langkah ini tetap dipertahankan di pipeline sebagai pengaman, karena dataset diperbarui setiap hari dan tumpang tindih bisa muncul pada pembaruan berikutnya.

## 4. Outlier FRP

FRP sangat menceng ke kanan. IQR pada skala asli akan menandai terlalu banyak titik sebagai outlier, padahal kebakaran besar memang menghasilkan FRP tinggi. Karena itu IQR dihitung pada **log10(FRP)**:

In [9]:
lazy_stage = pl.scan_parquet(STAGE_PATH)

q = lazy_stage.filter(pl.col("frp") > 0).select(
    pl.col("frp").quantile(0.25).alias("q1"), pl.col("frp").quantile(0.75).alias("q3")
).collect().row(0, named=True)
iqr = q["q3"] - q["q1"]
batas_iqr_asli = (q["q1"] - 1.5 * iqr, q["q3"] + 1.5 * iqr)
batas_iqr_log = cleaning.log_frp_iqr_bounds(lazy_stage)

def hitung_outlier(batas):
    lo, hi = batas
    return lazy_stage.filter(pl.col("frp") > 0).select(
        ((pl.col("frp") < lo) | (pl.col("frp") > hi)).mean() * 100
    ).collect().item()

pl.DataFrame({
    "metode": ["IQR skala asli", "IQR skala log10"],
    "batas_bawah_MW": [round(batas_iqr_asli[0], 2), round(batas_iqr_log[0], 2)],
    "batas_atas_MW": [round(batas_iqr_asli[1], 2), round(batas_iqr_log[1], 2)],
    "persen_outlier": [round(hitung_outlier(batas_iqr_asli), 2), round(hitung_outlier(batas_iqr_log), 2)],
})

metode,batas_bawah_MW,batas_atas_MW,persen_outlier
str,f64,f64,f64
"""IQR skala asli""",-6.79,17.37,8.83
"""IQR skala log10""",0.32,58.21,1.35


In [10]:
# Agregasi dulu (histogram log10 FRP, 80 bin) baru divisualisasikan, bukan 6 juta titik.
histogram = (
    lazy_stage.filter(pl.col("frp") > 0)
    .select((pl.col("frp").log10() * 20).floor().truediv(20).alias("log_frp"))
    .group_by("log_frp").len()
    .sort("log_frp")
    .collect()
)

fig = go.Figure(go.Bar(
    x=10 ** histogram["log_frp"], y=histogram["len"], width=10 ** histogram["log_frp"] * 0.1,
    marker_color="#c2410c", hovertemplate="FRP ≈ %{x:.2f} MW<br>%{y:,} deteksi<extra></extra>",
))
for batas, label in zip(batas_iqr_log, ["batas bawah IQR-log", "batas atas IQR-log"]):
    fig.add_vline(x=batas, line_dash="dash", line_color="#1e3a8a",
                  annotation_text=f"{label}: {batas:.2f} MW", annotation_position="top")
fig.update_layout(
    title="Distribusi FRP (skala log) dan batas outlier IQR-log",
    xaxis=dict(type="log", title="Fire Radiative Power (MW, skala log)"),
    yaxis_title="Jumlah deteksi", template="plotly_white", height=420, bargap=0,
)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.write_html(FIGURES_DIR / "m2_distribusi_frp.html", include_plotlyjs="cdn")
fig.show()

In [11]:
# Apakah outlier atas terkonsentrasi di wilayah dan tahun kebakaran besar (indikasi nilai nyata)?
connection.execute(f"""
    SELECT province, COUNT(*) AS outlier_atas,
           ROUND(MEDIAN(frp), 1) AS median_frp_outlier,
           MODE(year(acq_date)) AS tahun_terbanyak
    FROM read_parquet('{STAGE_PATH.as_posix()}')
    WHERE frp > {batas_iqr_log[1]}
    GROUP BY province ORDER BY outlier_atas DESC LIMIT 8
""").pl()

province,outlier_atas,median_frp_outlier,tahun_terbanyak
str,i64,f64,i64
"""West Kalimantan""",15490,81.2,2026
"""Central Kalimantan""",12338,81.4,2026
"""South Sumatra""",8883,85.4,2015
"""Riau""",5951,85.3,2014
"""Papua""",4710,82.6,2026
"""Jambi""",3702,87.8,2019
"""East Kalimantan""",2977,80.1,2015
"""South Kalimantan""",1770,80.6,2026


**Keputusan:** batas IQR-log (≈ 0,32–58 MW) menandai ±1,4% deteksi sebagai outlier, jauh lebih masuk akal daripada IQR skala asli. Outlier atas terkonsentrasi di provinsi dengan lahan gambut luas (Kalbar, Kalteng, Sumsel, Riau) dan di tahun kebakaran besar (2014, 2015, 2019, 2026). Artinya ini kebakaran nyata, bukan kesalahan sensor. Maka:
- nilai `frp` asli **dipertahankan** (tidak dihapus/diubah),
- `frp_outlier` = penanda (flag),
- `frp_winsor` = FRP yang di-*clip* ke batas IQR-log (windsorization), dipakai untuk rata-rata atau visualisasi yang sensitif terhadap nilai ekstrem.

Tidak ada batas domain yang dilanggar (FRP ≥ 0, maksimum ±1.951 MW masih wajar untuk piksel 375 m).

## 5. Anomali Sensor dan Tingkat Keyakinan

In [12]:
connection.execute(f"""
    SELECT confidence,
           COUNT(*)                                          AS jumlah,
           COUNT(*) FILTER (WHERE bright_ti4 >= 367)         AS ti4_saturasi,
           COUNT(*) FILTER (WHERE bright_ti4 < bright_ti5)   AS ti4_lt_ti5,
           ROUND(MEDIAN(frp), 2)                             AS median_frp
    FROM read_parquet('{STAGE_PATH.as_posix()}')
    GROUP BY confidence ORDER BY jumlah DESC
""").pl()

confidence,jumlah,ti4_saturasi,ti4_lt_ti5,median_frp
str,i64,i64,i64,f64
"""nominal""",5522529,0,0,4.25
"""low""",369635,3506,6863,5.41
"""high""",194966,194966,16,14.52


- **Saturasi I4 (367 K)** adalah batas atas sensor. Piksel yang sangat panas "terpotong" di nilai ini. Tabel di atas menunjukkan **seluruh** deteksi `high` adalah piksel tersaturasi, artinya kategori `high` pada VIIRS 375 m memang terkait dengan saturasi I4, bukan kesalahan data. Nilainya dipertahankan dan diberi flag `ti4_saturasi`, supaya `bright_ti4` tidak dipakai sebagai ukuran suhu absolut.
- **`ti4 < ti5`** tidak lazim untuk api (kanal 3,7 µm seharusnya lebih panas). 6.863 dari 6.879 baris berkeyakinan `low`, jadi diberi flag `anomali_ti4_lt_ti5`.
- **Confidence `low`** (±6%, semuanya siang hari) tidak dihapus, tapi diberi flag `andal = False`. Analisis di Milestone 3 bisa memilih memakai semua deteksi atau hanya yang andal.

## 6. Standardisasi dan Pemetaan Wilayah

Nama provinsi dipetakan ke nama resmi bahasa Indonesia, lalu ditambah atribut `pulau` dan `zona_waktu`. Waktu UTC dikonversi ke waktu lokal (WIB/WITA/WIT). Pemetaan memakai 34 provinsi sesuai batas geoBoundaries pada sumber data (sebelum pemekaran Papua 2022).

In [13]:
provinsi_data = connection.execute("SELECT DISTINCT province FROM hotspot_raw").pl()
tidak_terpetakan = provinsi_data.join(cleaning.PROVINSI, on="province", how="anti")
assert tidak_terpetakan.is_empty(), f"Provinsi belum dipetakan: {tidak_terpetakan}"
print(f"{provinsi_data.height} provinsi di data, semuanya terpetakan.")
cleaning.PROVINSI.group_by("pulau", "zona_waktu").agg(pl.len().alias("jumlah_provinsi")).sort("pulau")

34 provinsi di data, semuanya terpetakan.


pulau,zona_waktu,jumlah_provinsi
str,str,u32
"""Bali & Nusa Tenggara""","""WITA""",3
"""Jawa""","""WIB""",6
"""Kalimantan""","""WITA""",3
"""Kalimantan""","""WIB""",2
"""Maluku & Papua""","""WIT""",4
"""Sulawesi""","""WITA""",6
"""Sumatera""","""WIB""",10


## 7. Menjalankan Pipeline Cleaning (Tahap 2 — Polars Lazy)

`cleaning.pipeline_cleaning` merangkai standardisasi → penanganan nilai → konversi `Enum` dalam satu `LazyFrame`. Hasilnya ditulis dengan `sink_parquet`, sehingga data diproses streaming dan tidak dimuat utuh ke RAM.

In [14]:
pipeline = cleaning.pipeline_cleaning(lazy_stage, batas_iqr_log, frp_medians)
print(pipeline.explain().replace(str(Path(tempfile.gettempdir())), "<tmp>")[:1500])

simple π 24/24 ["acq_date", "acq_datetime", ... 22 other columns]
   WITH_COLUMNS:
   [((col("frp") < 0.324088) | (col("frp") > 58.205466)).alias("frp_outlier"), col("frp").clip([dyn float: 0.3240881168537487, dyn float: 58.2054664118173]).alias("frp_winsor"), col("acq_date").dt.year().alias("tahun"), col("acq_date").dt.month().alias("bulan"), col("satellite").strict_cast(Enum([...])), col("confidence").strict_cast(Enum([...])), col("daynight").strict_cast(Enum([...])), col("processing").strict_cast(Enum([...])), col("zona_waktu").strict_cast(Enum([...])), col("pulau").strict_cast(Categorical), col("provinsi").strict_cast(Categorical)] 
    simple π 20/20 ["acq_date", "acq_datetime", ... 18 other columns]
       WITH_COLUMNS:
       [col("frp").fill_null([col("frp_median_grup")])] 
        LEFT JOIN:
        LEFT PLAN ON: [col("satellite"), col("daynight"), col("province")]
           WITH_COLUMNS:
           [(col("acq_datetime") + 0.dt.duration([0, col("zona_waktu").replace_strict([[

In [15]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
mulai = time.perf_counter()
pipeline.sink_parquet(CLEAN_PATH)
print(f"Durasi tahap 2 : {time.perf_counter() - mulai:.1f} detik")
print(f"Output         : {CLEAN_PATH.relative_to(ROOT)} ({CLEAN_PATH.stat().st_size / 1e6:.1f} MB)")

lazy_bersih = pl.scan_parquet(CLEAN_PATH)
lazy_bersih.collect_schema()

Durasi tahap 2 : 18.2 detik
Output         : data/processed/hotspots_clean.parquet (119.7 MB)


Schema([('acq_date', Date),
        ('acq_datetime', Datetime(time_unit='ns', time_zone=None)),
        ('acq_datetime_lokal', Datetime(time_unit='us', time_zone=None)),
        ('tahun', Int32),
        ('bulan', Int8),
        ('satellite', Enum(categories=['Suomi NPP', 'NOAA-20'])),
        ('processing', Enum(categories=['SP', 'NRT'])),
        ('daynight', Enum(categories=['D', 'N'])),
        ('latitude', Float64),
        ('longitude', Float64),
        ('province', String),
        ('provinsi', Categorical),
        ('pulau', Categorical),
        ('zona_waktu', Enum(categories=['WIB', 'WITA', 'WIT'])),
        ('confidence', Enum(categories=['low', 'nominal', 'high'])),
        ('andal', Boolean),
        ('frp', Float64),
        ('frp_winsor', Float64),
        ('frp_outlier', Boolean),
        ('frp_imputasi', Boolean),
        ('bright_ti4', Float64),
        ('bright_ti5', Float64),
        ('ti4_saturasi', Boolean),
        ('anomali_ti4_lt_ti5', Boolean)])

In [16]:
lazy_bersih.head(3).collect()

acq_date,acq_datetime,acq_datetime_lokal,tahun,bulan,satellite,…,frp_outlier,frp_imputasi,bright_ti4,bright_ti5,ti4_saturasi,anomali_ti4_lt_ti5
date,datetime[ns],datetime[μs],i32,i8,enum,…,bool,bool,f64,f64,bool,bool
2019-09-05,2019-09-05 06:10:00,2019-09-05 13:10:00,2019,9,"""NOAA-20""",…,false,false,343.27,307.49,false,false
2019-09-05,2019-09-05 06:10:00,2019-09-05 13:10:00,2019,9,"""NOAA-20""",…,false,false,349.74,307.94,false,false
2019-09-05,2019-09-05 06:10:00,2019-09-05 13:10:00,2019,9,"""NOAA-20""",…,false,false,347.87,307.34,false,false


## 8. Validasi Data Bersih

Seluruh aturan data mentah ditambah aturan untuk kolom hasil cleaning dijalankan dengan DuckDB. `assert_all_pass` akan **menggagalkan eksekusi notebook** jika ada satu aturan saja yang dilanggar, sehingga data yang lolos ke Milestone 3 dijamin memenuhi aturan.

In [17]:
connection.execute(f"CREATE OR REPLACE VIEW hotspot_bersih AS SELECT * FROM read_parquet('{CLEAN_PATH.as_posix()}')")

laporan_sesudah = validation.run_rules(
    connection,
    "hotspot_bersih",
    validation.RAW_RULES + validation.clean_rules(batas_iqr_log),
    unique_key=cleaning.KEY_COLUMNS,
)
validation.assert_all_pass(laporan_sesudah)
print(f"Semua {laporan_sesudah.height} aturan lulus.")
laporan_sesudah

Semua 19 aturan lulus.


aturan,dimensi,deskripsi,pelanggaran,persen,lulus
str,str,str,i64,f64,bool
"""kolom_kunci_lengkap""","""completeness""","""Satelit, waktu, dan koordinat wajib terisi""",0,0.0,true
"""frp_terisi""","""completeness""","""FRP wajib terisi""",0,0.0,true
"""frp_positif""","""validity""","""FRP deteksi api harus > 0 MW""",0,0.0,true
"""bright_ti5_wajar""","""validity""","""Suhu kecerahan I5 <= 150 K tidak realistis""",0,0.0,true
"""koordinat_di_indonesia""","""validity""","""Koordinat berada di kotak wilayah Indonesia""",0,0.0,true
"""satelit_valid""","""validity""","""Hanya Suomi NPP / NOAA-20""",0,0.0,true
"""confidence_valid""","""validity""","""Confidence hanya low/nominal/high""",0,0.0,true
"""daynight_valid""","""validity""","""Siang/malam hanya D/N""",0,0.0,true
"""processing_valid""","""validity""","""Status pemrosesan hanya SP/NRT""",0,0.0,true


### Cek silang dengan ringkasan resmi dataset (*accuracy*)

Pembuat dataset menyediakan `daily_province.csv` berisi jumlah deteksi Suomi NPP per hari per provinsi. Hasil agregasi data bersih harus sama persis dengan file itu. Ini membuktikan cleaning tidak menghilangkan atau menggandakan deteksi.

In [18]:
if DAILY_REF_PATH.exists():
    cek_silang = connection.execute(f"""
        WITH hasil AS (
            SELECT acq_date AS date, province, COUNT(*) AS n
            FROM hotspot_bersih WHERE satellite = 'Suomi NPP'
            GROUP BY ALL
        ),
        referensi AS (SELECT * FROM read_csv('{DAILY_REF_PATH.as_posix()}'))
        SELECT COUNT(*)                                                         AS pasangan_hari_provinsi,
               COUNT(*) FILTER (WHERE COALESCE(n, 0) <> COALESCE(hotspots_snpp, 0)) AS tidak_cocok,
               SUM(n)::BIGINT                                                   AS total_hasil,
               SUM(hotspots_snpp)::BIGINT                                       AS total_referensi
        FROM referensi FULL JOIN hasil USING (date, province)
    """).pl()
    assert cek_silang["tidak_cocok"].item() == 0, "Hasil cleaning tidak cocok dengan referensi"
    display(cek_silang)
else:
    print("daily_province.csv tidak ada di data/raw/ — cek silang dilewati (opsional).")

pasangan_hari_provinsi,tidak_cocok,total_hasil,total_referensi
i64,i64,i64,i64
120653,0,4140721,4140721


## 9. Ringkasan Sebelum vs Sesudah

In [19]:
ringkasan = (
    laporan_sebelum.select("aturan", "dimensi", pl.col("pelanggaran").alias("sebelum"))
    .join(laporan_sesudah.select("aturan", pl.col("pelanggaran").alias("sesudah")), on="aturan", how="full", coalesce=True)
    .with_columns(pl.col("dimensi").fill_null("(aturan data bersih)"))
)
ringkasan

aturan,dimensi,sebelum,sesudah
str,str,i64,i64
"""kolom_kunci_lengkap""","""completeness""",0,0
"""frp_terisi""","""completeness""",0,0
"""frp_positif""","""validity""",189,0
"""bright_ti5_wajar""","""validity""",3,0
"""koordinat_di_indonesia""","""validity""",0,0
"""satelit_valid""","""validity""",0,0
"""confidence_valid""","""validity""",0,0
"""daynight_valid""","""validity""",0,0
"""processing_valid""","""validity""",0,0


In [20]:
flag = lazy_bersih.select(
    pl.len().alias("total_baris"),
    pl.col("frp_imputasi").sum(),
    pl.col("frp_outlier").sum(),
    pl.col("ti4_saturasi").sum(),
    pl.col("anomali_ti4_lt_ti5").sum(),
    (~pl.col("andal")).sum().alias("tidak_andal"),
    pl.col("bright_ti5").null_count().alias("ti5_null"),
).collect()
flag.transpose(include_header=True, header_name="kolom", column_names=["jumlah"])

kolom,jumlah
str,u32
"""total_baris""",6087130
"""frp_imputasi""",189
"""frp_outlier""",82175
"""ti4_saturasi""",198472
"""anomali_ti4_lt_ti5""",6879
"""tidak_andal""",369635
"""ti5_null""",3


In [21]:
ukuran = pl.DataFrame({
    "file": ["mentah (data/raw)", "bersih (data/processed)"],
    "baris": [jumlah_awal, lazy_bersih.select(pl.len()).collect().item()],
    "kolom": [len(lazy_raw.collect_schema()), len(lazy_bersih.collect_schema())],
    "ukuran_MB": [round(RAW_PATH.stat().st_size / 1e6, 1), round(CLEAN_PATH.stat().st_size / 1e6, 1)],
})
STAGE_PATH.unlink(missing_ok=True)  # hapus file sementara tahap 1
ukuran

file,baris,kolom,ukuran_MB
str,i64,i64,f64
"""mentah (data/raw)""",6087130,12,133.4
"""bersih (data/processed)""",6087130,24,119.7


## Catatan Milestone 2

**Dimensi kualitas yang paling bermasalah**
- *Completeness temporal*: 62 hari-satelit tanpa data (55 Suomi NPP, 7 NOAA-20). Penyebab terbesar adalah gangguan Suomi NPP 27 Jul – 10 Agu 2022 yang jatuh di musim kebakaran. Kalau diisi nol, tren 2022 akan terlihat lebih rendah dari kenyataannya.
- *Validity/accuracy* sinyal sensor: 198 ribu piksel I4 tersaturasi, 6,9 ribu piksel `ti4 < ti5`, 370 ribu deteksi `low` (6%), dan FRP yang sangat menceng (skewness 14).
- Duplikat dan null eksplisit tidak ada. Data sumber sudah cukup rapi, sehingga pekerjaan utama adalah **menandai** masalah, bukan membuang baris.

**Strategi yang dipilih**
| Masalah | Strategi | Baris terdampak |
|---|---|---|
| `frp = 0` | Jadi null → imputasi median per (satelit, siang/malam, provinsi), flag `frp_imputasi` | 189 |
| `bright_ti5 ≤ 150 K` | Jadi null, tanpa imputasi | 3 |
| Hari tanpa deteksi | Tidak diisi nol, disimpan di `hari_tanpa_deteksi.parquet` untuk Milestone 3 | 62 hari |
| Duplikat / NRT tergantikan SP | Deduplikasi kunci deteksi (DuckDB), SP diprioritaskan | 0 (pengaman) |
| Outlier FRP | IQR pada log10(FRP) → flag `frp_outlier` + kolom `frp_winsor`, nilai asli dipertahankan | 82.175 (1,35%) |
| Saturasi I4, `ti4 < ti5` | Flag `ti4_saturasi`, `anomali_ti4_lt_ti5` | 198.472 / 6.879 |
| Confidence `low` | Flag `andal = False` (tidak dihapus) | 369.635 |
| Konsistensi | Nama provinsi Indonesia, pulau, zona waktu, waktu lokal, `Date`/`Enum` | semua |

**Perubahan jumlah baris:** 6.087.130 → 6.087.130 (tidak ada baris dibuang). Kolom bertambah dari 12 menjadi 24 (atribut wilayah, waktu lokal, flag kualitas). File bersih justru lebih kecil (120 MB vs 133 MB) berkat tipe `Enum`/`Categorical`.

**Validasi:** 19 aturan (completeness, validity, consistency, uniqueness) dijalankan otomatis dengan DuckDB, dan notebook akan berhenti dengan error jika ada aturan yang dilanggar. Sebelum cleaning ada 2 aturan yang gagal (`frp_positif`, `bright_ti5_wajar`), setelah cleaning semuanya lulus. Cek silang dengan ringkasan resmi `daily_province.csv` cocok 100% (120.653 pasangan hari-provinsi, total 4.140.721 deteksi Suomi NPP).

**Catatan efisiensi:** deduplikasi dengan `LazyFrame.unique()` pada 6 juta baris terkena *out-of-memory* di VM RAM 4 GB, jadi tahap ini dipindah ke DuckDB (`memory_limit = 1GB`, spill ke disk). Tahap transformasi Polars dieksekusi streaming lewat `sink_parquet`. Seluruh notebook berjalan ±1 menit tanpa memuat data utuh ke RAM.

**Untuk Milestone 3:** pakai `data/processed/hotspots_clean.parquet`. Tren jangka panjang memakai Suomi NPP saja (dua satelit tidak dijumlahkan), dan hari pada `hari_tanpa_deteksi.parquet` diperlakukan sebagai data hilang. Kolom `andal` dan `frp_winsor` tersedia untuk analisis yang sensitif terhadap noise atau outlier.